## Imports

In [0]:
from pyspark.sql.functions import col, lower, when, lit, current_timestamp
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
import boto3

## Read Parameters

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


In [0]:
env = get_env_prefix()
env = 'p'
print(env)

In [0]:
Businessdate = dbutils.widgets.get("Businessdate")

if not Businessdate:
    PreviousBusinessDay = spark.sql(f"""                    
                            select PreviousBusinessDay
                            from `{env}-cdh`.reference.ref_calendar
                            where CalendarDate = cast(getdate() as date)
                        """).first()[0]
    Businessdate = spark.sql(f"""                    
                            select PreviousBusinessDay
                            from `{env}-cdh`.reference.ref_calendar
                            where CalendarDate = '{PreviousBusinessDay}'
                        """).first()[0]


## Read Tables

In [0]:
case_master = spark.sql(f"""
                        select distinct
                              vend_code
                              , Extra_Segment
                              , extc_code
                              , Sub_Segment
                              , debt_code
                              , case_code
                              , que_desc
                              , process_type
                              , emp_code_assignment
                              , cust_code
                              , extc_code
                              , strategy_step_id
                              , case_relation_type_code
                              , case_currently_open_flag
                        from `{env}-cdh`.landed.qc_case_master
                        where BusinessDate = '{Businessdate}'
                      """)

case_master.createOrReplaceTempView("case_master")

In [0]:
customer = spark.sql(f"""
                        select distinct
                                Segment
                              , channel as customer_channel
                              , cust_code
                        from `{env}-cdh`.landed.qc_customer
                        where BusinessDate = '{Businessdate}'
                     """)

customer.createOrReplaceTempView("customer")

In [0]:
Portfolio = spark.sql(f"""
                        select distinct vendor_code_qualco
                        from `{env}-cdh`.primary.Portfolio
                        where BusinessDate = '{Businessdate}'
                     """)

Portfolio.createOrReplaceTempView("Portfolio")

In [0]:
strategy_step = spark.sql(f"""
                              select distinct strategy_step_id
                              from `{env}-cdh`.landed.qc_strategy_step
                              where BusinessDate = '{Businessdate}'
                           """)

strategy_step.createOrReplaceTempView("strategy_step")

In [0]:
employees = spark.sql(f"""
                        select distinct 
                                  emp_code
                                , team_id
                                , dep_code
                                , team_name
                        from `{env}-cdh`.landed.qc_employees
                        where BusinessDate = '{Businessdate}'
                      """)

employees.createOrReplaceTempView("employees")

In [0]:
assignment_queue_setup = spark.sql(f"""
                                    select distinct 
                                               assignment_mapping
                                             , queue_desc
                                    from `{env}-cdh`.landed.params_assignment_queue_setup
                                    where BusinessDate = '{Businessdate}'
                                 """)

assignment_queue_setup.createOrReplaceTempView("assignment_queue_setup")

In [0]:
external_companies = spark.sql(f"""
                                 select distinct 
                                       EXTC_COMMENT
                                       , extc_code
                                 from `{env}-cdh`.landed.qc_external_companies
                                 where BusinessDate = '{Businessdate}'
                              """)

external_companies.createOrReplaceTempView("external_companies")

In [0]:
Team_L3869_Segments = spark.sql(f"""
                                 select  *
                                 from `d-cdh`.reference.team_l_3869_segments
                              """)

Team_L3869_Segments.createOrReplaceTempView("Team_L3869_Segments")

In [0]:
team =  spark.sql(f"""
                    select TEA_NAME, TEA_CODE, channel_id
                    from `{env}-cdh`.landed.qc_team 
                    where BusinessDate = '{Businessdate}'
                    and TEA_NAME = 'Mercury CA Team N3869'
                """)

team.createOrReplaceTempView("team")
display(team)

In [0]:
ca_dialer_team_mercury_3869 =  spark.sql(f"""
                                            select TEA_CODE 
                                            from team
                                        """).collect()[0][0]
display(ca_dialer_team_mercury_3869)

In [0]:
ca_dialer_channel_mercury_3869 = spark.sql(f"""
                                            SELECT channel_id
                                            from team
                                        """).collect()[0][0]
print(ca_dialer_channel_mercury_3869)

In [0]:
ChannelCollectionAgents = spark.sql(f"""
                                        select distinct dep_code
                                        from `{env}-cdh`.landed.qc_department
                                        where BusinessDate = '{Businessdate}'
                                        and dep_desc = 'Collection Agents'
                                    """).collect()[0][0]
print(ChannelCollectionAgents)
# display(ChannelCollectionAgents)

## Create Select Queries

#### Inserting into qc.case_assignment for Mercury and rest 3rd party portfolios

In [0]:
s1 = spark.sql(f"""
                    SELECT DISTINCT
                           cm.vend_code
                         , cm.debt_code
                         , cm.case_code
                         , cm.que_desc
                         , c.Segment
                         , cm.Extra_Segment
                         , cm.process_type
                         , cm.emp_code_assignment
                         , CASE
                              WHEN ec.extc_code IN (76,81,83,87,88,104,116) THEN ec.extc_code
                              ELSE cm.extc_code
                           END  AS extc_code
                        , CASE
                              WHEN lower(c.customer_channel) IN ('external law firms 27','external law firms 28')
                                   AND cm.process_type = 'MRC_Offer'
                                   AND cm.vend_code = 6
                                   THEN 'DNC_Offer'
                              WHEN (cm.process_type IN ('MRC_N3869','MRC_Offer') AND ec.extc_code IN (76,81,83,87,88,104,116))
                                   OR (coalesce(cm.process_type,'') = ''
                                        AND ec.extc_code IN (76,81,83,87,88,104,116)
                                        AND cm.emp_code_assignment IS NULL
                                        AND upper(c.segment) NOT LIKE '%CP')                  
                                   THEN 'LF_Offer'
                              WHEN cm.vend_code = 6
                                   AND cm.extc_code IN (81,87,105,106)
                                   AND lower(cm.process_type) LIKE '%collection%'           
                                   THEN 'Collection'
                              WHEN cm.vend_code = 6
                                   AND cm.que_desc = 'MRC_Coll_Jud_Escalation'
                                   AND cm.extc_code IN (81,87)                               
                                   THEN 'Collection_Judicial_Escalation'
                              WHEN cm.emp_code_assignment IS NOT NULL THEN 'CA/FA/Sme Officer'
                              WHEN aqs.assignment_mapping IS NOT NULL
                                   AND aqs.assignment_mapping <> 'CA Dialer'                 
                                   THEN aqs.assignment_mapping
                              WHEN l3869.team IS NOT NULL
                                   AND cm.vend_code = 6
                                   AND coalesce(cm.process_type,'') IN ('','MRC_Judicial','Mercury 3869 Judicial')
                                   THEN CASE
                                             WHEN l3869.segment = 'L3869 Court Approved'
                                                  AND cm.Sub_Segment IN ('Leverage A','Leverage B')
                                                  THEN 'Passive_Court Approved'
                                             ELSE l3869.team
                                        END
                              WHEN aqs.assignment_mapping = 'CA Dialer'
                                   AND coalesce(cm.process_type,'') IN ('','MRC_Judicial','Mercury 3869 Judicial')
                                   AND cm.vend_code = 6                                      THEN aqs.assignment_mapping

                              WHEN lower(c.segment) LIKE '%cp'
                                   AND cm.vend_code = 6
                                   AND coalesce(cm.process_type,'') IN ('','MRC_Judicial','Mercury 3869 Judicial')
                                   THEN 'CA Dialer'
                              WHEN lower(c.customer_channel) = 'external law firms 7' THEN 'Unhandled'
                              WHEN cm.process_type IN ('MRC_Judicial','Mercury 3869 Judicial')
                                   AND c.customer_channel <> ''                              
                                   THEN 'Unhandled'
                              ELSE 'Unhandled'
                           END  AS assignment_process
                         , CASE
                              WHEN lower(c.customer_channel) IN ('external law firms 27','external law firms 28')
                                   AND cm.process_type = 'MRC_Offer'
                                   AND cm.vend_code = 6                                    
                                   THEN -1
                              WHEN (cm.process_type IN ('MRC_N3869','MRC_Offer') AND ec.extc_code IN (76,81,83,87,88,104,116))
                                   OR (coalesce(cm.process_type,'') = ''
                                        AND ec.extc_code IN (76,81,83,87,88,104,116)
                                        AND cm.emp_code_assignment IS NULL
                                        AND upper(c.segment) NOT LIKE '%CP')                 
                                   THEN -1
                              WHEN cm.vend_code = 6
                                   AND cm.extc_code IN (81,87,105,106)
                                   AND lower(cm.process_type) LIKE '%collection%'            
                                   THEN -1
                              WHEN cm.vend_code = 6
                                   AND cm.que_desc = 'MRC_Coll_Jud_Escalation'
                                   AND cm.extc_code IN (81,87)                               
                                   THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL THEN e.emp_code
                              ELSE -1
                           END   AS agent_id
                         , CASE
                              WHEN lower(c.customer_channel) IN ('external law firms 27','external law firms 28')
                                   AND cm.process_type = 'MRC_Offer'
                                   AND cm.vend_code   = 6                                    
                                   THEN -1
                              WHEN (cm.process_type IN ('MRC_N3869','MRC_Offer')
                                        AND ec.extc_code IN (76,81,83,87,88,104,116))
                                   OR (coalesce(cm.process_type,'') = ''
                                        AND ec.extc_code IN (76,81,83,87,88,104,116)
                                        AND cm.emp_code_assignment IS NULL
                                        AND upper(c.segment) NOT LIKE '%CP')                  
                                        THEN -1
                              WHEN cm.vend_code = 6
                                   AND cm.extc_code IN (81,87,105,106)
                                   AND lower(cm.process_type) LIKE '%collection%'            
                                   THEN -1
                              WHEN cm.vend_code = 6
                                   AND cm.que_desc = 'MRC_Coll_Jud_Escalation'
                                   AND cm.extc_code IN (81,87)                               
                                   THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL THEN e.team_id

                              WHEN aqs.assignment_mapping IS NOT NULL
                                   AND aqs.assignment_mapping <> 'CA Dialer' 
                                   THEN -1
                              WHEN l3869.team IS NOT NULL
                                   AND cm.vend_code = 6
                                   AND coalesce(cm.process_type,'') IN ('','MRC_Judicial','Mercury 3869 Judicial')
                                   THEN {ca_dialer_team_mercury_3869}
                              ELSE -1
                           END  AS team_id
                         , CASE
                              WHEN lower(c.customer_channel) IN ('external law firms 27','external law firms 28')
                                   AND cm.process_type = 'MRC_Offer'
                                   AND cm.vend_code   = 6                                    
                                   THEN ec.extc_code
                              WHEN (cm.process_type IN ('MRC_N3869','MRC_Offer') AND ec.extc_code IN (76,81,83,87,88,104,116))
                                   OR (coalesce(cm.process_type,'') = ''
                                        AND ec.extc_code IN (76,81,83,87,88,104,116)
                                        AND cm.emp_code_assignment IS NULL
                                        AND upper(c.segment) NOT LIKE '%CP')                  
                                   THEN ec.extc_code
                              WHEN cm.vend_code = 6
                                   AND cm.extc_code IN (81,87,105,106)
                                   AND lower(cm.process_type) LIKE '%collection%'            
                                   THEN cm.extc_code
                              WHEN cm.vend_code = 6
                                   AND cm.que_desc = 'MRC_Coll_Jud_Escalation'
                                   AND cm.extc_code IN (81,87)                               
                                   THEN cm.extc_code
                              WHEN cm.emp_code_assignment IS NOT NULL THEN e.dep_code
                              WHEN aqs.assignment_mapping IS NOT NULL
                                   AND aqs.assignment_mapping <> 'CA Dialer'                 
                                   THEN -1
                              WHEN l3869.team IS NOT NULL
                                   AND cm.vend_code = 6
                                   AND coalesce(cm.process_type,'') IN ('','MRC_Judicial','Mercury 3869 Judicial')
                                   THEN {ca_dialer_channel_mercury_3869}
                              ELSE -1
                           END AS channel_id
                         , CASE
                              WHEN lower(c.customer_channel) IN ('external law firms 27','external law firms 28')
                                   AND cm.process_type = 'MRC_Offer'
                                   AND cm.vend_code = 6                                    
                                   THEN -1
                              WHEN (cm.process_type IN ('MRC_N3869','MRC_Offer') AND cm.extc_code IN (76,81,83,87,88,104,116))
                                   OR (coalesce(cm.process_type,'') = ''
                                        AND cm.extc_code IN (76,81,83,87,88,104,116)
                                        AND cm.emp_code_assignment IS NULL
                                        AND upper(c.segment) NOT LIKE '%CP')                  
                                   THEN -1
                              WHEN cm.vend_code = 6
                                   AND cm.extc_code IN (81,87,105,106)
                                   AND lower(cm.process_type) LIKE '%collection%'            
                                   THEN -1
                              WHEN cm.vend_code = 6
                                   AND cm.que_desc = 'MRC_Coll_Jud_Escalation'
                                   AND cm.extc_code IN (81,87)                               
                                   THEN cm.extc_code
                              WHEN cm.emp_code_assignment IS NOT NULL THEN coalesce(cm.extc_code, -1)
                              WHEN aqs.assignment_mapping IS NOT NULL
                                   AND aqs.assignment_mapping <> 'CA Dialer'                 
                                   THEN -1
                              WHEN l3869.team IS NOT NULL
                                   AND cm.vend_code = 6
                                   AND coalesce(cm.process_type,'') IN ('','MRC_Judicial','Mercury 3869 Judicial')
                                   THEN coalesce(cm.extc_code, -1)
                              WHEN aqs.assignment_mapping = 'CA Dialer'
                                   AND coalesce(cm.process_type,'') IN ('','MRC_Judicial','Mercury 3869 Judicial')
                                   AND cm.vend_code = 6                                    
                                   THEN coalesce(cm.extc_code, -1)
                              WHEN c.segment LIKE '%CP'
                                   AND cm.vend_code = 6
                                   AND coalesce(cm.process_type,'') IN ('','MRC_Judicial','Mercury 3869 Judicial')
                                   THEN coalesce(cm.extc_code, -1)
                              WHEN cm.extc_code = 82 THEN coalesce(cm.extc_code, -1)
                              WHEN cm.process_type IN ('MRC_Judicial','Mercury 3869 Judicial')
                                   AND c.customer_channel <> ''                              
                                   THEN coalesce(cm.extc_code, -1)
                              ELSE -1
                           END AS legal_channel_id
                         , CAST(DATE_ADD('{Businessdate}', 1) AS DATE) AS update_date
                         , c.customer_channel
                         , CAST(NULL AS SMALLINT) AS final_bucket
                    FROM case_master cm
                    INNER JOIN customer c
                    ON c.cust_code = cm.cust_code
                    INNER JOIN Portfolio p
                    ON p.vendor_code_qualco = cm.vend_code
                    LEFT JOIN strategy_step ss
                    ON ss.strategy_step_id = cm.strategy_step_id
                    LEFT JOIN employees e 
                    ON e.emp_code = cm.emp_code_assignment
                    LEFT JOIN assignment_queue_setup aqs
                    ON aqs.queue_desc = cm.que_desc
                    LEFT JOIN Team_L3869_Segments l3869
                    ON (
                         c.Segment LIKE concat(l3869.segment, '%')
                         AND l3869.extra_segment = cm.Extra_Segment
                         AND c.segment <> 'L3869 Rest for service call_CP'
                         )
                    OR (
                         c.Segment LIKE concat(l3869.segment, '%')
                         AND l3869.extra_segment = 'Any Value'
                         )
                    LEFT JOIN external_companies ec
                    ON lower(c.customer_channel) = lower(ec.EXTC_COMMENT)
                    WHERE cm.vend_code IN (6,7,8,9)
                    AND cm.case_relation_type_code = 2090
                    AND cm.case_currently_open_flag = true
               """)

s1.createOrReplaceTempView("s1")

#### Inserting into qc.case_assignment for Orbit portfolio

In [0]:
s2 = spark.sql(f"""
                    SELECT DISTINCT
                           cm.vend_code
                         , cm.debt_code
                         , cm.case_code
                         , cm.que_desc
                         , c.Segment
                         , cm.Extra_Segment
                         , cm.process_type
                         , cm.emp_code_assignment
                         , CASE
                              WHEN lower(c.customer_channel) IN ('external law firms 27','external law firms 28')
                                   AND cm.process_type = 'Offer'                  
                                   THEN ec.extc_code
                              WHEN lower(c.customer_channel) IN ('external law firms 1','external law firms 5', 'external law firms 12','external law firms 13')
                                   AND cm.process_type = 'Offer'                  
                                   THEN ec.extc_code
                              ELSE cm.extc_code
                           END AS extc_code
                         , CASE
                              WHEN lower(c.customer_channel) IN ('external law firms 27','external law firms 28')
                                   AND cm.process_type = 'Offer'                  
                                   THEN 'DNC_Offer'
                              WHEN lower(c.customer_channel) IN
                                        ('external law firms 1','external law firms 5',
                                        'external law firms 12','external law firms 13',
                                        'external law firms 26','external law firms 30')
                                   AND cm.process_type = 'Offer'                  
                                   THEN 'LF_Offer'
                              WHEN lower(cm.process_type) LIKE '%collection%'
                                   AND cm.extc_code IS NOT NULL                   
                                   THEN 'Collection'
                              WHEN cm.emp_code_assignment IS NOT NULL THEN 'CA/FA/Sme Officer'
                              WHEN c.segment LIKE '%1st Lien%'
                                   OR c.segment LIKE 'L3869%'
                                   OR c.segment LIKE 'L4605%'                        
                                   THEN 'Passive_Segment'
                              WHEN c.segment = '' THEN 'Passive_No_Segment'
                              WHEN aqs.assignment_mapping IS NOT NULL THEN aqs.assignment_mapping
                              WHEN cm.que_desc IN ('Orbit Fulfilled Settlements', 'Orbit Fulfilled Settlements to Recall') THEN 'Fulfilled Settlment / Paid Off'
                              WHEN c.customer_channel NOT LIKE '%Financial Advisors%'
                                   AND coalesce(ec.extc_code,0) NOT IN (105,106)
                                   AND c.customer_channel <> ''
                                   THEN 'CA Dialer'
                              ELSE 'Unhandled'
                           END AS assignment_process
                         , CASE
                              WHEN ec.extc_code IN (105,106)
                                   AND cm.process_type = 'Offer'                  
                                   THEN -1
                              WHEN ec.extc_code IN (76,81,87,88,104,116)
                                   AND cm.process_type = 'Offer'                  
                                   THEN -1
                              WHEN lower(cm.process_type) LIKE '%collection%'
                                   AND cm.extc_code IS NOT NULL                   
                                   THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL THEN e.emp_code
                              ELSE -1
                           END AS agent_id
                         , CASE
                              WHEN ec.extc_code IN (105,106)
                                   AND cm.process_type = 'Offer'
                                   THEN -1
                              WHEN ec.extc_code IN (76,81,87,88,104,116)
                                   AND cm.process_type = 'Offer'
                                   THEN -1
                              WHEN lower(cm.process_type) LIKE '%collection%'
                                   AND cm.extc_code IS NOT NULL
                                   THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL THEN e.team_id
                              ELSE -1
                           END AS team_id
                         , CASE
                              WHEN ec.extc_code IN (105,106)
                                   AND cm.process_type = 'Offer'
                                   THEN ec.extc_code
                              WHEN ec.extc_code IN (76,81,87,88,104,116)
                                   AND cm.process_type = 'Offer'
                                   THEN ec.extc_code
                              WHEN lower(cm.process_type) LIKE '%collection%'
                                   AND cm.extc_code IS NOT NULL
                                   THEN cm.extc_code
                              WHEN cm.emp_code_assignment IS NOT NULL THEN e.dep_code
                              ELSE -1
                           END AS channel_id
                         , CASE
                              WHEN ec.extc_code IN (105,106)
                                   AND cm.process_type = 'Offer'
                                   THEN -1
                              WHEN ec.extc_code IN (76,81,87,88,104,116)
                                   AND cm.process_type = 'Offer'
                                   THEN -1
                              ELSE
                                   CASE
                                        WHEN cm.extc_code IN (105,106) THEN -1
                                        ELSE coalesce(cm.extc_code,-1)
                                   END
                           END AS legal_channel_id
                         , CAST(DATE_ADD('{Businessdate}',1) AS DATE) AS update_date
                         , c.customer_channel
                         , CAST(NULL AS SMALLINT) AS final_bucket
                    FROM case_master cm
                    INNER JOIN customer c
                    ON c.cust_code = cm.cust_code
                    LEFT JOIN strategy_step ss  
                    ON ss.strategy_step_id = cm.strategy_step_id
                    LEFT JOIN employees e   
                    ON e.emp_code = cm.emp_code_assignment
                    LEFT JOIN assignment_queue_setup aqs 
                    ON aqs.queue_desc = cm.que_desc
                    LEFT JOIN external_companies ec  
                    ON lower(c.customer_channel) = lower(ec.EXTC_COMMENT)
                    WHERE cm.vend_code = 13
                    AND cm.case_relation_type_code = 2090
                    AND cm.case_currently_open_flag = true
               """)
               
s2.createOrReplaceTempView("s2")

In [0]:
s2 = (
    s2.withColumn(
        "legal_channel_id",
        when(
            (col("assignment_process") == "Fulfilled Settlment / Paid Off") &
            (col("legal_channel_id") != -1) &
            (col("vend_code") == 13),
            -1
        ).otherwise(col("legal_channel_id"))
    )
)

s2.createOrReplaceTempView("s2")

#### Inserting into qc.case_assignment for Light portfolio

In [0]:
s3 = spark.sql(f"""
                    SELECT DISTINCT
                           cm.vend_code
                         , cm.debt_code
                         , cm.case_code
                         , cm.que_desc
                         , c.Segment
                         , cm.Extra_Segment
                         , cm.process_type
                         , cm.emp_code_assignment
                         , cm.extc_code
                         , CASE
                              WHEN cm.process_type LIKE '%Offer%' 
                                   AND cm.extc_code IN (105,106) 
                                   THEN 'DNC_Offer'
                              WHEN cm.process_type LIKE '%Offer%' THEN 'LF_Offer'
                              WHEN cm.process_type LIKE '%Collection%' THEN 'Collection'
                              WHEN cm.emp_code_assignment IS NOT NULL 
                                   AND e.team_name IS NOT NULL
                                   THEN 'CA/FA/Sme Officer'
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE 'Wholesale%' 
                                        OR c.segment LIKE 'Stated Guaranteed%' 
                                        OR c.segment LIKE '%1st Lien%' 
                                        OR c.segment LIKE 'L3869%' 
                                        OR c.segment LIKE 'L4605%')
                                   THEN 'Passive_Segment'
                              WHEN c.segment IS NULL
                                   AND cm.emp_code_assignment IS NULL
                                   THEN 'Passive_No Segment'
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   THEN aqs.assignment_mapping
                              WHEN cm.que_desc IN ('Light Offer CA & Jud Assign','Light Offer CA','Light Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')
                                   THEN 'CA Dialer'
                              ELSE 'Unhandled'
                           END AS assignment_process
                         , CASE
                              WHEN cm.process_type LIKE '%Offer%' AND cm.extc_code IN (105,106) THEN -1
                              WHEN cm.process_type LIKE '%Offer%' THEN -1
                              WHEN cm.process_type LIKE '%Collection%' THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL AND e.team_name IS NOT NULL THEN cm.emp_code_assignment
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE 'Wholesale%' 
                                        OR c.segment LIKE 'Stated Guaranteed%' 
                                        OR c.segment LIKE '%1st Lien%' 
                                        OR c.segment LIKE 'L3869%' 
                                        OR c.segment LIKE 'L4605%')
                                   THEN -1
                              WHEN c.segment IS NULL
                                   AND cm.emp_code_assignment IS NULL
                                   THEN -1
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   THEN -1
                              WHEN cm.que_desc IN ('Light Offer CA & Jud Assign','Light Offer CA','Light Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')
                                   THEN -1
                              ELSE -1
                           END AS agent_id
                         , CASE
                              WHEN cm.process_type LIKE '%Offer%' AND cm.extc_code IN (105,106) THEN -1
                              WHEN cm.process_type LIKE '%Offer%' THEN -1
                              WHEN cm.process_type LIKE '%Collection%' THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL AND e.team_name IS NOT NULL THEN e.team_id
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE 'Wholesale%' 
                                        OR c.segment LIKE 'Stated Guaranteed%' 
                                        OR c.segment LIKE '%1st Lien%'
                                        OR c.segment LIKE 'L3869%' 
                                        OR c.segment LIKE 'L4605%')
                                   THEN -1
                              WHEN c.segment IS NULL 
                                   AND cm.emp_code_assignment IS NULL
                                   THEN -1
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   THEN -1
                              WHEN cm.que_desc IN ('Light Offer CA & Jud Assign','Light Offer CA','Light Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')
                                   THEN -1
                              ELSE -1
                           END AS team_id
                         , CASE
                              WHEN cm.process_type LIKE '%Offer%' 
                                   AND cm.extc_code IN (105,106)
                                   THEN CASE 
                                             WHEN cm.process_type LIKE 'Offer%' THEN coalesce(cm.extc_code,-1) 
                                             ELSE -1 
                                        END
                              WHEN lower(cm.process_type) LIKE '%offer%' THEN coalesce(cm.extc_code,-1)
                              WHEN cm.process_type LIKE '%Collection%'
                                   THEN CASE
                                             WHEN cm.process_type LIKE 'Collection%' THEN coalesce(cm.extc_code,-1) 
                                             ELSE -1 
                                        END
                              WHEN cm.emp_code_assignment IS NOT NULL 
                                   AND e.team_name IS NOT NULL
                                   THEN e.dep_code
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE 'Wholesale%' 
                                        OR c.segment LIKE 'Stated Guaranteed%' 
                                        OR c.segment LIKE '%1st Lien%' 
                                        OR c.segment LIKE 'L3869%' 
                                        OR c.segment LIKE 'L4605%')
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND (cm.process_type LIKE 'Collection%' OR cm.process_type LIKE '%Offer%')
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN c.segment IS NULL
                                   AND cm.emp_code_assignment IS NULL
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND (cm.process_type LIKE 'Collection%' OR cm.process_type LIKE '%Offer%')
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   AND NOT (aqs.queue_desc LIKE '%Fulfilled%'
                                            OR aqs.queue_desc LIKE '%Paid_Off%'
                                            OR aqs.queue_desc LIKE '%Εξοφλήσεις%')
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND (cm.process_type LIKE 'Collection%' OR cm.process_type LIKE '%Offer%')
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   AND (aqs.queue_desc LIKE '%Fulfilled%'
                                        OR aqs.queue_desc LIKE '%Paid_Off%'
                                        OR aqs.queue_desc LIKE '%Εξοφλήσεις%')
                                   THEN ec.extc_code
                              WHEN cm.que_desc IN ('Light Offer CA & Jud Assign','Light Offer CA','Light Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')
                                   THEN {ChannelCollectionAgents}
                              ELSE -1
                           END AS channel_id
                         , CASE
                              WHEN lower(cm.process_type) LIKE '%offer%' AND cm.extc_code IN (105,106) THEN -1
                              WHEN lower(cm.process_type) LIKE '%offer%' THEN -1
                              WHEN cm.process_type LIKE '%Collection%' THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL
                                   AND e.team_name IS NOT NULL
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND cm.process_type NOT LIKE 'Collection%'
                                                  AND cm.process_type NOT LIKE '%Offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE 'Wholesale%' 
                                        OR c.segment LIKE 'Stated Guaranteed%' 
                                        OR c.segment LIKE '%1st Lien%'
                                        OR c.segment LIKE 'L3869%' 
                                        OR c.segment LIKE 'L4605%')
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND cm.process_type NOT LIKE 'Collection%'
                                                  AND cm.process_type NOT LIKE '%Offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN c.segment IS NULL
                                   AND cm.emp_code_assignment IS NULL
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND cm.process_type NOT LIKE 'Collection%'
                                                  AND cm.process_type NOT LIKE '%Offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND cm.process_type NOT LIKE 'Collection%'
                                                  AND cm.process_type NOT LIKE '%Offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.que_desc IN ('Light Offer CA & Jud Assign','Light Offer CA','Light Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND cm.process_type NOT LIKE 'Collection%'
                                                  AND cm.process_type NOT LIKE '%Offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              ELSE -1
                           END AS legal_channel_id
                         , CAST(DATE_ADD('{Businessdate}',1) AS DATE) AS update_date
                         , c.customer_channel
                         , CAST(NULL AS SMALLINT) AS final_bucket
                    FROM case_master cm
                    INNER JOIN customer c   
                    ON c.cust_code = cm.cust_code
                    LEFT JOIN strategy_step ss  
                    ON ss.strategy_step_id = cm.strategy_step_id
                    LEFT JOIN employees e   
                    ON e.emp_code = cm.emp_code_assignment
                    LEFT JOIN assignment_queue_setup aqs 
                    ON aqs.queue_desc = cm.que_desc
                    LEFT JOIN external_companies ec  
                    ON lower(c.customer_channel) = lower(ec.EXTC_COMMENT)
                    WHERE cm.vend_code = 20
                    AND cm.case_relation_type_code = 2090
                    AND cm.case_currently_open_flag = true
               """)

s3.createOrReplaceTempView("s3")

#### Inserting into qc.case_assignment for Cell portfolio

In [0]:
s4 = spark.sql(f"""
                    SELECT DISTINCT
                           cm.vend_code
                         , cm.debt_code
                         , cm.case_code
                         , cm.que_desc
                         , c.Segment
                         , cm.Extra_Segment
                         , cm.process_type
                         , cm.emp_code_assignment
                         , cm.extc_code
                         , CASE
                              WHEN lower(cm.process_type) LIKE '%offer%'
                                   AND cm.extc_code IN (105,106)
                                   THEN 'DNC_Offer'
                              WHEN lower(cm.process_type) LIKE '%offer%' THEN 'LF_Offer'
                              WHEN lower(cm.process_type) LIKE '%collection%' THEN 'Collection'
                              WHEN cm.emp_code_assignment IS NOT NULL 
                                   AND e.team_name IS NOT NULL
                                   THEN 'CA/FA/Sme Officer'
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE '%1st Lien%' 
                                        OR c.segment LIKE 'L4605%' 
                                        OR c.segment LIKE 'L3869 Accepted%' 
                                        OR c.segment LIKE 'L3869 Pending%' 
                                        OR c.segment LIKE 'L3869 Rejected >10k%')
                                   THEN 'Passive_Segment'
                              WHEN c.segment IS NULL
                                   AND cm.emp_code_assignment IS NULL                 
                                   THEN 'Passive_No Segment'
                              WHEN cm.emp_code_assignment IS NULL            
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL                                                                
                                   THEN aqs.assignment_mapping
                              WHEN cm.que_desc IN ('Cell Offer CA & Jud Assign','Cell Offer CA','Cell Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR lower(cm.process_type) LIKE '%judicial%')                     
                                   THEN 'CA Dialer'
                              ELSE 'Unhandled'
                           END AS assignment_process
                         , CASE
                              WHEN lower(cm.process_type) LIKE '%offer%'     
                                   AND cm.extc_code IN (105,106)                      
                                   THEN -1
                              WHEN lower(cm.process_type) LIKE '%offer%' THEN -1
                              WHEN lower(cm.process_type) LIKE '%collection%' THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL 
                                   AND e.team_name IS NOT NULL
                                   THEN cm.emp_code_assignment
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE '%1st Lien%' 
                                        OR c.segment LIKE 'L4605%' 
                                        OR c.segment LIKE 'L3869 Accepted%' 
                                        OR c.segment LIKE 'L3869 Pending%' 
                                        OR c.segment LIKE 'L3869 Rejected >10k%')                   
                                   THEN -1
                              WHEN c.segment IS NULL 
                                   AND cm.emp_code_assignment IS NULL
                                   THEN -1
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   THEN -1
                              WHEN cm.que_desc IN ('Cell Offer CA & Jud Assign','Cell Offer CA','Cell Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR lower(cm.process_type) LIKE '%judicial%')
                                   THEN -1
                              ELSE -1
                           END AS agent_id
                         , CASE
                              WHEN lower(cm.process_type) LIKE '%offer%' AND cm.extc_code IN (105,106) THEN -1
                              WHEN lower(cm.process_type) LIKE '%offer%' THEN -1
                              WHEN lower(cm.process_type) LIKE '%collection%' THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL AND e.team_name IS NOT NULL THEN e.team_id
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE '%1st Lien%' 
                                        OR c.segment LIKE 'L4605%' 
                                        OR c.segment LIKE 'L3869 Accepted%' 
                                        OR c.segment LIKE 'L3869 Pending%' 
                                        OR c.segment LIKE 'L3869 Rejected >10k%')   
                                   THEN -1
                              WHEN c.segment IS NULL AND cm.emp_code_assignment IS NULL THEN -1
                              WHEN cm.emp_code_assignment IS NULL 
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   THEN -1
                              WHEN cm.que_desc IN ('Cell Offer CA & Jud Assign','Cell Offer CA','Cell Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR lower(cm.process_type) LIKE '%judicial%') THEN -1
                              ELSE -1
                           END AS team_id
                         , CASE
                              WHEN lower(cm.process_type) LIKE '%offer%'     
                                   AND cm.extc_code IN (105,106)
                                   THEN 
                                        CASE 
                                             WHEN lower(cm.process_type) LIKE 'offer%' THEN coalesce(cm.extc_code,-1) 
                                             ELSE -1 
                                        END
                              WHEN lower(cm.process_type) LIKE '%offer%' THEN coalesce(cm.extc_code,-1)
                              WHEN lower(cm.process_type) LIKE '%collection%'
                                   THEN 
                                        CASE 
                                             WHEN lower(cm.process_type) LIKE 'collection%' THEN coalesce(cm.extc_code,-1) 
                                             ELSE -1 
                                        END
                              WHEN cm.emp_code_assignment IS NOT NULL 
                                   AND e.team_name IS NOT NULL
                                   THEN e.dep_code
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE '%1st Lien%' 
                                        OR c.segment LIKE 'L4605%' 
                                        OR c.segment LIKE 'L3869 Accepted%' 
                                        OR c.segment LIKE 'L3869 Pending%' 
                                        OR c.segment LIKE 'L3869 Rejected >10k%')
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND (lower(cm.process_type) LIKE 'collection%' OR lower(cm.process_type) LIKE '%offer%')
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN c.segment IS NULL AND cm.emp_code_assignment IS NULL
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND (lower(cm.process_type) LIKE 'collection%' OR lower(cm.process_type) LIKE '%offer%')
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   AND NOT (aqs.queue_desc LIKE '%Fulfilled%' 
                                         OR aqs.queue_desc LIKE '%Paid_Off%' 
                                         OR aqs.queue_desc LIKE '%Εξοφλήσεις%')
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND (lower(cm.process_type) LIKE 'collection%' OR lower(cm.process_type) LIKE '%offer%')
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.emp_code_assignment IS NULL            
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   AND (aqs.queue_desc LIKE '%Fulfilled%' 
                                     OR aqs.queue_desc LIKE '%Paid_Off%' 
                                     OR aqs.queue_desc LIKE '%Εξοφλήσεις%')
                                   THEN ec.extc_code
                              WHEN cm.que_desc IN ('Cell Offer CA & Jud Assign','Cell Offer CA','Cell Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR lower(cm.process_type) LIKE '%judicial%')
                                   THEN {ChannelCollectionAgents}
                              ELSE -1
                           END AS channel_id
                         , CASE
                              WHEN lower(cm.process_type) LIKE '%offer%' AND cm.extc_code IN (105,106) THEN -1
                              WHEN lower(cm.process_type) LIKE '%offer%' THEN -1
                              WHEN lower(cm.process_type) LIKE '%collection%' THEN -1
                              WHEN cm.emp_code_assignment IS NOT NULL AND e.team_name IS NOT NULL
                                   THEN 
                                        CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND lower(cm.process_type) NOT LIKE 'collection%'
                                                  AND lower(cm.process_type) NOT LIKE '%offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.emp_code_assignment IS NULL
                                   AND (c.segment LIKE '%1st Lien%' 
                                        OR c.segment LIKE 'L4605%' 
                                        OR c.segment LIKE 'L3869 Accepted%' 
                                        OR c.segment LIKE 'L3869 Pending%' 
                                        OR c.segment LIKE 'L3869 Rejected >10k%') 
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND lower(cm.process_type) NOT LIKE 'collection%'
                                                  AND lower(cm.process_type) NOT LIKE '%offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN c.segment IS NULL 
                                   AND cm.emp_code_assignment IS NULL
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND lower(cm.process_type) NOT LIKE 'collection%'
                                                  AND lower(cm.process_type) NOT LIKE '%offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.emp_code_assignment IS NULL
                                   AND e.team_name IS NULL
                                   AND aqs.queue_desc IS NOT NULL
                                   THEN CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND lower(cm.process_type) NOT LIKE 'collection%'
                                                  AND lower(cm.process_type) NOT LIKE '%offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              WHEN cm.que_desc IN ('Cell Offer CA & Jud Assign','Cell Offer CA','Cell Offer VIP CA')
                                   AND cm.emp_code_assignment IS NULL
                                   AND (cm.process_type IS NULL OR lower(cm.process_type) LIKE '%judicial%')
                                   THEN 
                                        CASE
                                             WHEN cm.process_type IS NOT NULL
                                                  AND lower(cm.process_type) NOT LIKE 'collection%'
                                                  AND lower(cm.process_type) NOT LIKE '%offer%'
                                                  THEN coalesce(cm.extc_code,-1)
                                             ELSE -1
                                        END
                              ELSE -1
                           END AS legal_channel_id
                         , CAST(DATE_ADD('{Businessdate}',1) AS DATE) AS update_date
                         , c.customer_channel
                         , CAST(NULL AS SMALLINT) AS final_bucket
                    FROM case_master cm
                    INNER JOIN customer c   
                    ON c.cust_code = cm.cust_code
                    LEFT JOIN strategy_step ss  
                    ON ss.strategy_step_id = cm.strategy_step_id
                    LEFT JOIN employees e
                    ON e.emp_code = cm.emp_code_assignment
                    LEFT JOIN assignment_queue_setup aqs 
                    ON aqs.queue_desc = cm.que_desc
                    LEFT JOIN external_companies ec  
                    ON lower(c.customer_channel) = lower(ec.EXTC_COMMENT)
                    WHERE  cm.vend_code = 30
                    AND cm.case_relation_type_code = 2090
                    AND cm.case_currently_open_flag = true
               """)

s4.createOrReplaceTempView("s4")

#### Inserting into qc.case_assignment for Pearl portfolio

In [0]:
s5 = spark.sql(f"""
                    SELECT DISTINCT
                      cm.vend_code
                    , cm.debt_code
                    , cm.case_code
                    , cm.que_desc
                    , c.Segment
                    , cm.Extra_Segment
                    , cm.process_type
                    , cm.emp_code_assignment
                    , cm.extc_code
                    , CASE
                         WHEN cm.process_type LIKE '%Offer%'      
                              AND cm.extc_code IN (105,106)                      
                              THEN 'DNC_Offer'
                         WHEN cm.process_type LIKE '%Offer%' 
                              THEN 'LF_Offer'
                         WHEN cm.process_type LIKE '%Collection%'                                                   
                              THEN 'Collection'
                         WHEN cm.emp_code_assignment IS NOT NULL  
                              AND e.team_name IS NOT NULL                        
                              THEN 'CA/FA/Sme Officer'
                         WHEN cm.emp_code_assignment IS NULL   
                              AND (c.segment LIKE 'L3869 Accepted%' 
                                   OR c.segment LIKE 'L3869 Pending%' 
                                   OR c.segment LIKE 'L3869 Rejected >10k%') 
                              THEN 'Passive_Segment'
                         WHEN c.segment IS NULL                 
                              AND cm.emp_code_assignment IS NULL                   
                              THEN 'Passive_No Segment'
                         WHEN cm.emp_code_assignment IS NULL     
                              AND e.team_name IS NULL
                              AND aqs.queue_desc IS NOT NULL
                              THEN aqs.assignment_mapping
                         WHEN cm.emp_code_assignment IS NULL 
                              AND (cm.que_desc LIKE '%Offer CA & Jud Assign' 
                                   OR cm.que_desc LIKE '%Offer CA' 
                                   OR cm.que_desc LIKE '%Offer VIP CA')
                              AND (cm.process_type IS NULL 
                                   OR cm.process_type LIKE '%Judicial%')                     
                                   THEN 'CA Dialer'
                         ELSE 'Unhandled'
                      END AS assignment_process                    
                    , CASE
                         WHEN cm.process_type LIKE '%Offer%'      
                              AND cm.extc_code IN (105,106)                      
                              THEN -1
                         WHEN cm.process_type LIKE '%Offer%'                                                         
                              THEN -1
                         WHEN cm.process_type LIKE '%Collection%'                                                   
                              THEN -1
                         WHEN cm.emp_code_assignment IS NOT NULL  
                              AND e.team_name IS NOT NULL                        
                              THEN cm.emp_code_assignment
                         WHEN cm.emp_code_assignment IS NULL  
                              AND (c.segment LIKE 'L3869 Accepted%' 
                                   OR c.segment LIKE 'L3869 Pending%' 
                                   OR c.segment LIKE 'L3869 Rejected >10k%')             
                              THEN -1
                         WHEN c.segment IS NULL                 
                              AND cm.emp_code_assignment IS NULL                   
                              THEN -1
                         WHEN cm.emp_code_assignment IS NULL     
                              AND e.team_name IS NULL
                              AND aqs.queue_desc IS NOT NULL                                                         
                              THEN -1
                         WHEN cm.emp_code_assignment IS NULL
                              AND (cm.que_desc LIKE '%Offer CA & Jud Assign' 
                                   OR cm.que_desc LIKE '%Offer CA' 
                                   OR cm.que_desc LIKE '%Offer VIP CA')
                              AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')                     
                              THEN -1
                         ELSE -1
                      END AS agent_id                   
                    , CASE
                         WHEN lower(cm.process_type) LIKE '%offer%' 
                              AND cm.extc_code IN (105,106)                    
                              THEN -1
                         WHEN lower(cm.process_type) LIKE '%offer%'                                                  
                              THEN -1
                         WHEN cm.process_type LIKE '%Collection%'                                                   
                              THEN -1
                         WHEN cm.emp_code_assignment IS NOT NULL  
                              AND e.team_name IS NOT NULL                        
                              THEN e.team_id
                         WHEN cm.emp_code_assignment IS NULL 
                              AND (c.segment LIKE 'L3869 Accepted%' 
                                   OR c.segment LIKE 'L3869 Pending%' 
                                   OR c.segment LIKE 'L3869 Rejected >10k%')             
                              THEN -1
                         WHEN c.segment IS NULL                 
                              AND cm.emp_code_assignment IS NULL                   
                              THEN -1
                         WHEN cm.emp_code_assignment IS NULL     
                              AND e.team_name IS NULL
                              AND aqs.queue_desc IS NOT NULL                                                         
                              THEN -1
                         WHEN cm.emp_code_assignment IS NULL 
                              AND (cm.que_desc LIKE '%Offer CA & Jud Assign' 
                                   OR cm.que_desc LIKE '%Offer CA' 
                                   OR cm.que_desc LIKE '%Offer VIP CA')
                              AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')                     
                              THEN -1
                         ELSE -1
                      END AS team_id                  
                    , CASE
                         WHEN lower(cm.process_type) LIKE '%offer%' 
                              AND cm.extc_code IN (105,106)
                              THEN 
                                   CASE 
                                        WHEN lower(cm.process_type) LIKE 'offer%'      
                                             THEN coalesce(cm.extc_code,-1) 
                                        ELSE -1 
                                   END
                         WHEN lower(cm.process_type) LIKE '%offer%' THEN coalesce(cm.extc_code,-1)
                         WHEN cm.process_type LIKE '%Collection%'
                              THEN 
                                   CASE 
                                        WHEN cm.process_type LIKE 'Collection%'      
                                             THEN coalesce(cm.extc_code,-1) 
                                        ELSE -1 
                                   END
                         WHEN cm.emp_code_assignment IS NOT NULL  
                              AND e.team_name IS NOT NULL                        
                              THEN e.dep_code
                         WHEN cm.emp_code_assignment IS NULL
                              AND (c.segment LIKE 'L3869 Accepted%' 
                                   OR c.segment LIKE 'L3869 Pending%' 
                                   OR c.segment LIKE 'L3869 Rejected >10k%') 
                              THEN CASE
                                        WHEN cm.process_type IS NOT NULL
                                             AND (cm.process_type LIKE 'Collection%' OR cm.process_type LIKE '%Offer%')
                                             THEN coalesce(cm.extc_code,-1)
                                        ELSE -1
                                   END
                         WHEN c.segment IS NULL
                              AND cm.emp_code_assignment IS NULL
                              THEN CASE
                                        WHEN cm.process_type IS NOT NULL
                                             AND (cm.process_type LIKE 'Collection%' OR cm.process_type LIKE '%Offer%')
                                             THEN coalesce(cm.extc_code,-1)
                                        ELSE -1
                                   END
                         WHEN cm.emp_code_assignment IS NULL     
                              AND e.team_name IS NULL
                              AND aqs.queue_desc IS NOT NULL
                              AND NOT (aqs.queue_desc LIKE '%Fulfilled%' 
                                   OR aqs.queue_desc LIKE '%Paid_Off%' 
                                   OR aqs.queue_desc LIKE '%Εξοφλήσεις%')
                              THEN CASE
                                        WHEN cm.process_type IS NOT NULL
                                             AND (cm.process_type LIKE 'Collection%' OR cm.process_type LIKE '%Offer%')
                                             THEN coalesce(cm.extc_code,-1)
                                        ELSE -1
                                   END
                         WHEN cm.emp_code_assignment IS NULL     
                              AND e.team_name IS NULL
                              AND aqs.queue_desc IS NOT NULL
                              AND (aqs.queue_desc LIKE '%Fulfilled%' 
                                   OR aqs.queue_desc LIKE '%Paid_Off%' 
                                   OR aqs.queue_desc LIKE '%Εξοφλήσεις%')
                              THEN ec.extc_code
                         WHEN (cm.que_desc LIKE '%Offer CA & Jud Assign' 
                                   OR cm.que_desc LIKE '%Offer CA' 
                                   OR cm.que_desc LIKE '%Offer VIP CA')
                              AND cm.emp_code_assignment IS NULL
                              AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')
                              THEN {ChannelCollectionAgents}
                         ELSE -1
                      END AS channel_id
                    , CASE
                         WHEN lower(cm.process_type) LIKE '%offer%' AND cm.extc_code IN (105,106) THEN -1
                         WHEN lower(cm.process_type) LIKE '%offer%' THEN -1
                         WHEN cm.process_type LIKE '%Collection%' THEN -1
                         WHEN cm.emp_code_assignment IS NOT NULL  
                              AND e.team_name IS NOT NULL
                              THEN CASE
                                        WHEN cm.process_type IS NOT NULL
                                             AND cm.process_type NOT LIKE 'Collection%'
                                             AND cm.process_type NOT LIKE '%Offer%'
                                             THEN coalesce(cm.extc_code,-1)
                                        ELSE -1
                                   END
                         WHEN cm.emp_code_assignment IS NULL
                              AND (c.segment LIKE 'L3869 Accepted%' 
                                   OR c.segment LIKE 'L3869 Pending%' 
                                   OR c.segment LIKE 'L3869 Rejected >10k%')
                              THEN CASE
                                        WHEN cm.process_type IS NOT NULL
                                             AND cm.process_type NOT LIKE 'Collection%'
                                             AND cm.process_type NOT LIKE '%Offer%'
                                             THEN coalesce(cm.extc_code,-1)
                                        ELSE -1
                                   END
                         WHEN c.segment IS NULL
                              AND cm.emp_code_assignment IS NULL
                              THEN CASE
                                        WHEN cm.process_type IS NOT NULL
                                             AND cm.process_type NOT LIKE 'Collection%'
                                             AND cm.process_type NOT LIKE '%Offer%'
                                             THEN coalesce(cm.extc_code,-1)
                                        ELSE -1
                                   END
                         WHEN cm.emp_code_assignment IS NULL     
                              AND e.team_name IS NULL
                              AND aqs.queue_desc IS NOT NULL
                              THEN CASE
                                        WHEN cm.process_type IS NOT NULL
                                             AND cm.process_type NOT LIKE 'Collection%'
                                             AND cm.process_type NOT LIKE '%Offer%'
                                             THEN coalesce(cm.extc_code,-1)
                                        ELSE -1
                                   END
                         WHEN (cm.que_desc LIKE '%Offer CA & Jud Assign' 
                              OR cm.que_desc LIKE '%Offer CA' 
                              OR cm.que_desc LIKE '%Offer VIP CA')
                              AND cm.emp_code_assignment IS NULL
                              AND (cm.process_type IS NULL OR cm.process_type LIKE '%Judicial%')
                              THEN CASE
                                        WHEN cm.process_type IS NOT NULL
                                             AND cm.process_type NOT LIKE 'Collection%'
                                             AND cm.process_type NOT LIKE '%Offer%'
                                             THEN coalesce(cm.extc_code,-1)
                                        ELSE -1
                                   END
                         ELSE -1
                      END AS legal_channel_id
                    , CAST(DATE_ADD('{Businessdate}',1) AS DATE) AS update_date
                    , c.customer_channel
                    , CAST(NULL AS SMALLINT) AS final_bucket
               FROM case_master cm
               INNER JOIN customer c   
               ON c.cust_code = cm.cust_code
               LEFT JOIN strategy_step ss  
               ON ss.strategy_step_id = cm.strategy_step_id
               LEFT JOIN employees e   
               ON e.emp_code = cm.emp_code_assignment
               LEFT JOIN assignment_queue_setup aqs 
               ON aqs.queue_desc = cm.que_desc
               LEFT JOIN external_companies ec  
               ON lower(c.customer_channel) = lower(ec.EXTC_COMMENT)
               WHERE cm.vend_code = 34
               AND cm.case_relation_type_code = 2090
               AND cm.case_currently_open_flag = true
          """)

s5.createOrReplaceTempView("s5")

## Create the Final Table

In [0]:
combined_df = s1.unionByName(s2)\
                .unionByName(s3)\
                .unionByName(s4)\
                .unionByName(s5)\
                .dropDuplicates()

combined_df.createOrReplaceTempView("combined_df")

In [0]:
combined_df = combined_df.withColumn("BusinessDate", lit(Businessdate))
combined_df = combined_df.withColumn("LoadingDateTime", lit(current_timestamp()))

combined_df.createOrReplaceTempView("combined_df")

## Compare Tables

#### Extract Original

In [0]:
case_assignment = spark.sql(f"""
                              select *
                              from `{env}-cdh`.landed.qc_case_assignment
                              where BusinessDate = '{Businessdate}'
                           """)

case_assignment.createOrReplaceTempView("case_assignment")

#### Define common cols

In [0]:
# Assume df1 and df2 are your two DataFrames
df1 = combined_df
df2 = case_assignment

# Step 1: Identify common columns dynamically
common_cols = list(set(df1.columns) & set(df2.columns))
common_cols.remove("LoadingDateTime")

print("Common columns being compared:", common_cols)

#### Make the Compares

In [0]:
# Step 2: Normalize values to avoid mismatches
df1_common = df1.select([lower(col(c)).alias(c) for c in common_cols]).dropDuplicates()
df2_common = df2.select([lower(col(c)).alias(c) for c in common_cols]).dropDuplicates()

# Check row counts
print("df1_common rows:", df1_common.count())
print("df2_common rows:", df2_common.count())

# Step 3: Compare
diff1 = df1_common.exceptAll(df2_common)
diff2 = df2_common.exceptAll(df1_common)

if diff1.isEmpty() and diff2.isEmpty():
    print("DataFrames are equal in common columns.")
else:
    print("Differences found:")
    display(diff1)
    display(diff2)
